# Tuần 1 · Dữ liệu, TF–IDF và Word2Vec
Đồ án NLP cá nhân · phiên bản 2.2. Đọc đề trước khi chạy. Cài thư viện một lần theo README nếu cần.
**C1** dùng train để khảo sát và validation để đo E1. **C2** huấn luyện Skip-gram và khảo sát một cặp token.

Dữ liệu: **Vietnamese AI Feedback OOD 3125**, giấy phép CC BY 4.0. Dùng các tệp CSV đã chia sẵn từ gói lớp. Xem README để đặt tệp và chạy. Tuần 1–2 chỉ đọc train và validation. Nhận tệp test của lớp vào tuần 3.

In [ ]:
import hashlib
import io
import json
from pathlib import Path
import re
import unicodedata

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score

LABELS = ["negative", "neutral", "positive"]
DATA_VERSION = "ood3125-de7a66b-group42-v1"
CHECKSUMS = {'train': '17e2029dcf20ba77ef95b62b77e4a0458eabea1d36cb72e0794f790340ea1201', 'validation': 'c8148193a9d14963fbc25837f5bb6277bb907d36d1811d2cc25ea6e935847d0c', 'test': '4bee1b48b6dc277742f4d9591c602a720d4a483646e57786df40bb47f2cb7ef7'}


def text_tokens(text):
    """Tokenize word-like spans after Unicode normalization and case folding."""
    return re.findall(r"\w+", unicodedata.normalize("NFC", text).casefold())


def read_split(split):
    filename = f"ood3125_{split}.csv"
    path = Path("data") / filename
    if not path.is_file():
        path = Path(filename)
    if not path.is_file():
        raise FileNotFoundError(f"Đặt {filename} trong data/ hoặc thư mục làm việc.")
    content = path.read_bytes()
    if hashlib.sha256(content).hexdigest() != CHECKSUMS[split]:
        raise ValueError(f"{filename} không đúng bản chia của lớp DCT1243.")
    frame = pd.read_csv(io.BytesIO(content))
    frame = frame.rename(columns={"sentence": "text"})
    frame["label"] = frame["label"].map(dict(enumerate(LABELS)))
    return frame


def scores(gold, predicted):
    per_label = f1_score(gold, predicted, labels=LABELS, average=None, zero_division=0)
    return {"accuracy": accuracy_score(gold, predicted), "macro_f1": per_label.mean(),
            **{f"f1_{label}": value for label, value in zip(LABELS, per_label)}}

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline


def fit_baseline(train):
    model = make_pipeline(
        TfidfVectorizer(tokenizer=text_tokens, token_pattern=None, lowercase=False,
                        ngram_range=(1, 2),
                        min_df=2, sublinear_tf=True),
        LogisticRegression(max_iter=2000, class_weight="balanced", random_state=42),
    )
    return model.fit(train["text"], train["label"])

from gensim.models import Word2Vec

MODEL_SEED = 42
EMBED_DIM = 100


def fit_word2vec(tokenized_train):
    return Word2Vec(tokenized_train, vector_size=EMBED_DIM, window=5, min_count=2,
                    sg=1, negative=5, epochs=20, seed=MODEL_SEED, workers=1)

In [ ]:
STUDENT_ID = ""
if not STUDENT_ID.isascii() or not STUDENT_ID.isdigit() or len(STUDENT_ID) < 3:
    raise ValueError("Điền MSSV gồm ít nhất ba chữ số rồi chạy lại ô này.")
PERSONAL_SEED = int(STUDENT_ID[-3:])
print("MSSV:", STUDENT_ID, "| Seed lấy mẫu:", PERSONAL_SEED)

In [ ]:
train, validation = read_split("train"), read_split("validation")
print("train / validation:", len(train), len(validation))
display(pd.DataFrame({name: data["label"].value_counts(normalize=True)
                     for name, data in [("train", train), ("validation", validation)]})
        .reindex(LABELS).round(4))
display(train[["row_id", "text", "label"]].sample(n=3, random_state=PERSONAL_SEED))

## C1 · Đo mô hình cơ sở E1
Khung E1 đã cho. Hoàn thành một lệnh gọi `scores`, in kết quả và ghi hai nhận xét có căn cứ trong báo cáo.
Nhận xét về phân bố nhãn và giải thích vì sao xem macro-F1 cùng F1 từng nhãn, dù ba nhãn khá cân bằng.
Ví dụ minh họa với F1 ba lớp là 0,80, 0,20 và 0,90 thì macro-F1 bằng 0,633.

In [ ]:
baseline = fit_baseline(train)
baseline_predictions = baseline.predict(validation["text"])
baseline_scores = None  # TODO: Call scores with gold and predicted labels.
if baseline_scores is None:
    raise NotImplementedError("Hoàn thành C1.")
display(pd.Series(baseline_scores).round(4))

## C2 · Huấn luyện và đọc kết quả Word2Vec
Tạo danh sách token từ câu train bằng `text_tokens()` và gọi `fit_word2vec`.
Chọn hai token có trong từ vựng. Đọc ba láng giềng của token thứ nhất, tính độ tương đồng cosin của cặp,
rồi giải thích một giới hạn khi dùng độ gần để suy ra cảm xúc.
Hàm chuẩn hóa Unicode, chuyển chữ về dạng thường và lấy các cụm ký tự chữ/số. Một token có thể là một âm tiết tiếng Việt.

**Thực hành trước khi làm mục này.** Xem [hướng dẫn Skip-gram](DCT1243_Skipgram_HuongDan.pdf) và chạy [notebook minh họa](DCT1243_Skipgram_SinhVien.ipynb) đi kèm. Nếu Colab không mở được liên kết tương đối, hãy mở hai tệp từ gói đã giải nén.

In [ ]:
tokenized_train = None  # TODO: Apply text_tokens to each training sentence.
if tokenized_train is None:
    raise NotImplementedError("Hoàn thành C2.")
word2vec = fit_word2vec(tokenized_train)
first, second = "", ""  # TODO: Choose two vocabulary tokens.

In [ ]:
if not (first in word2vec.wv and second in word2vec.wv and first != second):
    raise ValueError("Chọn hai token khác nhau, đều có trong từ vựng.")
print("Số token có vector:", len(word2vec.wv))
print("Cặp token:", first, second)
print("Láng giềng:", word2vec.wv.most_similar(first, topn=3))
print("Cosin:", round(float(word2vec.wv.similarity(first, second)), 4))